<a href="https://colab.research.google.com/github/interpretune/examples/blob/main/tutorials/publish/tuning/sae_lora_tuning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Adapting a Model to Its SAE: LoRA Around a Frozen Dictionary, in Stages

Splicing a sparse autoencoder into a language model costs next-token loss: the model runs on the SAE's reconstruction,
and whatever the SAE misses is gone. [Chen, Engels and Tegmark (2025)](https://arxiv.org/abs/2501.19406) leave the SAE
alone and adapt **the model around it**: they freeze the SAE, add LoRA adapters after it, and train them so that the
spliced model's next-token distribution matches the original model's. With

$$
\text{gap closed} = \frac{L_{SAE} - L}{L_{SAE} - L_{BASE}},
$$

where $L_{BASE}$ is the original model's loss, $L_{SAE}$ the loss with the SAE spliced in and $L$ the loss after
adapting, they report 30% to 55% of the gap closed on Gemma Scope SAEs, and that **one** adapted layer, the one right
after the SAE, reaches about 88% of what adapting every later layer does.

This notebook runs their comparison at a small scale, on GPT-2 small and a published SAE. Every arm is one schedule, and
every schedule entry is an address:

| Run | Phase entry | What it tunes |
|---|---|---|
| next layer | `role:adapter & layers:7` | the LoRA adapters of the block right after the SAE |
| all later layers | `role:adapter & layers:7:` | the LoRA adapters of every block after the SAE |
| the SAE | `role:latent:sae` | the SAE itself, tuned end to end from its published weights, as their comparison trains one |
| staged | the three above, in that order | next layer, then the later layers too, then the SAE, promoted from `artifact`, with the adapters held |

The question the staged run asks is what each stage adds over the one before it, against a control that spends the same
budget on one entry alone.

Requirements: `uv pip install -e ".[examples,tune]" --group git-deps`. The measured run below used one GPU; with the
parameters reduced the notebook runs on a CPU. GPT-2, the SAE (about 200 MB) and the text are public.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
STEPS_PER_EPOCH = 400  # each run takes EPOCHS epochs, so every run sees the same batches in the same order
EPOCHS = 3
BATCH_SIZE = 16  # blocks per batch
CONTEXT = 128  # tokens per block
TEST_BLOCKS = 512
VAL_BLOCKS = 64
SAE_LAYER = 6  # the residual stream after this block
RANK = 64
ADAPTER_LR = 5e-5  # the authors'
SAE_LR = 1e-5
RUNS = ("next layer", "all later layers", "the SAE", "staged")
SEED = 0

In [3]:
# @title Imports { display-mode: "form" }
import gc
import tempfile
import warnings
from collections import Counter
from functools import partial
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from finetuning_scheduler.core.checkpoint import load_payload
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM

import interpretune as it
from interpretune.config import SessionRunnerCfg
from interpretune.config.mixins import HFFromPretrainedConfig
from interpretune.harness.notebook_ui import display_notebook_table
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.latents import LatentSite, attach_latent_model
from interpretune.utils import ITInstantiationFeedbackWarning

# the configuration classes are composed as declared; this feedback repeats their whole contents for each session
warnings.filterwarnings("ignore", category=ITInstantiationFeedbackWarning)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

~/worktrees/interpretune-pub/src/interpretune/hub/trust.py:96: Refusing to execute analysis ops from 'speediedan/jlens_steering_ops': loading it runs code published by that repo inside this process, with your filesystem, network and Hugging Face token. Interpretune does not execute hub-resident code unless you opt in.
  Inspect it first: interpretune.hub.pull('speediedan/jlens_steering_ops') caches the repo without executing anything, so you can read the entrypoint the manifest names.
  Then opt in for this session: os.environ['IT_TRUST_REMOTE_CODE'] = '1' (or export IT_TRUST_REMOTE_CODE=1 before starting the process).
  Pin a revision when you pull so trusted code cannot change under you.
  Details: https://interpretune.org/en/latest/usage/hub_trust_posture.html


~/worktrees/interpretune-pub/src/interpretune/analysis/ops/dispatcher.py:1242: Alias 'ct_feature_intervention' already has an assigned op or alias so the alias specified by 'feature_intervention_forward' cannot be mapped to it
~/worktrees/interpretune-pub/src/interpretune/analysis/ops/dispatcher.py:1242: Alias 'model_forward' already has an assigned op or alias so the alias specified by 'model_fwd' cannot be mapped to it


## 1. Text, and the Two Losses Between Which the Gap Lies

The text is [pile-10k](https://huggingface.co/datasets/NeelNanda/pile-10k), 10,000 documents of the Pile, which the
paper trains and evaluates on. Documents 0-999 are the test blocks every result below is measured on, 1000-1999 the
validation blocks the runs report as they go, and the rest is for training. A block is GPT-2's end-of-text token and 127
tokens of text.

`TEACHER` is the original model, untouched: it gives $L_{BASE}$ and, in training, the distribution the spliced model is
trained to match.

In [4]:
# @title 1: The text and the datamodule { display-mode: "form" }
TEXT, TEXT_REVISION = "NeelNanda/pile-10k", "127bfedcd5047750df5ccf3a12979a47bfa0bafa"


class TextBlocks(it.ITDataModule):
    """Fixed-length blocks of text, packed in order; no padding, so there is no mask."""

    def prepare_data(self, target_model=None) -> None:
        load_dataset(TEXT, split="train", revision=TEXT_REVISION)

    def setup(self, stage=None, module=None, *args, **kwargs) -> None:
        if module is not None:
            self._module = module
        text = load_dataset(TEXT, split="train", revision=TEXT_REVISION)
        self.dataset = {
            "test": self.blocks(text.select(range(1000)), TEST_BLOCKS),
            "validation": self.blocks(text.select(range(1000, 2000)), VAL_BLOCKS),
            "train": self.blocks(text.select(range(2000, len(text))), EPOCHS * STEPS_PER_EPOCH * BATCH_SIZE),
        }

    def blocks(self, rows, n: int) -> Dataset:
        ids: list[int] = []
        for row in rows:
            ids += [*self.tokenizer(row["text"])["input_ids"], self.tokenizer.eos_token_id]
            if len(ids) >= n * (CONTEXT - 1):
                break
        else:
            raise ValueError(f"{len(rows)} documents hold {len(ids)} tokens, fewer than {n} blocks need")
        packed = torch.tensor(ids[: n * (CONTEXT - 1)]).view(n, CONTEXT - 1)
        bos = torch.full((n, 1), self.tokenizer.bos_token_id)
        return Dataset.from_dict({"input_ids": torch.cat([bos, packed], dim=1)}).with_format("torch")

    def loader(self, split: str, **kwargs) -> DataLoader:
        return DataLoader(self.dataset[split], batch_size=BATCH_SIZE, collate_fn=self.data_collator, **kwargs)

    def train_dataloader(self) -> DataLoader:  # one seeded order, so every run takes the same batches
        return self.loader("train", shuffle=True, generator=torch.Generator().manual_seed(SEED))

    def val_dataloader(self) -> DataLoader:
        return self.loader("validation")

    def test_dataloader(self) -> DataLoader:
        return self.loader("test")


def block_losses(logits: torch.Tensor, ids: torch.Tensor) -> torch.Tensor:
    """Each block's mean next-token cross-entropy."""
    return F.cross_entropy(logits[:, :-1].transpose(1, 2).float(), ids[:, 1:], reduction="none").mean(1)


TEACHER = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float32).to(DEVICE).eval().requires_grad_(False)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## 2. The Session: LoRA, and a Spliced SAE

The session composes the core adapter alone: GPT-2, LoRA adapters of rank 64 on the attention and MLP projections of the
blocks after the SAE (`lora_cfg`), and the SAE attached in `register_tunable_modules()`.

- **The SAE** is OpenAI's 32k-latent TopK residual SAE (32 active latents per token), loaded through sae_lens and
  spliced at `point:hook_resid_post & layers:6` with `op="replace"`, so the later blocks read its reconstruction and
  nothing else. Attached `frozen=True`, its tensors are `artifact`, which no stage can thaw until one promotes them. The
  run that tunes the SAE from its first step attaches it unfrozen instead, since a schedule's first phase promotes
  nothing.
- **The loss** is the paper's: the KL divergence $\mathrm{KL}(y \Vert \hat y)$ of the spliced model's next-token
  distribution $\hat y$ from the teacher's $y$ (the authors' code computes it as
  `F.kl_div(adapted_log_probs, base_probs)`).
- **Dropout is off** (`model_cfg`). GPT-2's configuration turns it on in training, where it would be noise the adapters
  are asked to match the teacher through.

In [5]:
# @title 2: The module and the session { display-mode: "form" }
SAE_RELEASE, SAE_ID = "gpt2-small-resid-post-v5-32k", f"blocks.{SAE_LAYER}.hook_resid_post"
AFTER = range(SAE_LAYER + 1, TEACHER.config.n_layer)  # the blocks that read the reconstruction


def load_sae() -> torch.nn.Module:
    from sae_lens import SAE

    loaded = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device=DEVICE)
    return (loaded[0] if isinstance(loaded, tuple) else loaded).float()


class AdaptedModule(torch.nn.Module):
    sae_frozen = True  # an artifact, until a stage promotes it

    def register_tunable_modules(self) -> None:
        super().register_tunable_modules()
        site = LatentSite(f"point:hook_resid_post & layers:{SAE_LAYER}", op="replace")
        attach_latent_model(self, "sae", load_sae(), site, frozen=self.sae_frozen)

    def training_step(self, batch, batch_idx):
        ids = batch["input_ids"]
        with torch.no_grad():
            target = F.log_softmax(TEACHER(input_ids=ids).logits, dim=-1)
        log_probs = F.log_softmax(self(input_ids=ids).logits, dim=-1)
        loss = F.kl_div(log_probs, target, reduction="none", log_target=True).sum(-1).mean()
        self.log("train_kl", loss)
        return loss

    def validation_step(self, batch, batch_idx, dataloader_idx: int = 0):
        self.log("val_loss", block_losses(self(input_ids=batch["input_ids"]).logits, batch["input_ids"]).mean())


class TunedSAEModule(AdaptedModule):
    sae_frozen = False  # `latent:sae` from the start: a schedule's first phase promotes nothing


def build_session(lr: float, module_cls: type = AdaptedModule) -> it.ITSession:
    session = it.ITSession(
        it.ITSessionConfig(
            adapter_ctx=(it.Adapter.core,),
            datamodule_cfg=it.ITDataModuleConfig(
                model_name_or_path="gpt2",
                task_name="pile_blocks",
                tokenizer_id_overrides={"pad_token_id": 50256},
                signature_columns=["input_ids"],
            ),
            module_cfg=it.ITConfig(
                model_name_or_path="gpt2",
                task_name="pile_blocks",
                model_cfg={"resid_pdrop": 0.0, "embd_pdrop": 0.0, "attn_pdrop": 0.0},
                hf_from_pretrained_cfg=HFFromPretrainedConfig(
                    pretrained_kwargs={"device_map": DEVICE, "dtype": "float32"},
                    model_head="transformers.GPT2LMHeadModel",
                    enable_input_require_grads=False,
                    lora_cfg={
                        "r": RANK,
                        "lora_alpha": RANK,
                        "target_modules": ["c_attn", "c_proj", "c_fc"],
                        "layers_to_transform": list(AFTER),
                        "layers_pattern": "h",
                        "fan_in_fan_out": True,
                    },
                ),
                optimizer_init={"class_path": "torch.optim.Adam", "init_args": {"lr": lr}},
            ),
            datamodule_cls=TextBlocks,
            module_cls=module_cls,
        )
    )
    it.it_init(**session)
    return session

## 3. The Runs

Each run trains for the same number of steps on the same batches. The staged run moves to its next stage after each
epoch; the other three spend the whole budget on one entry. A start from the spliced, unadapted model is the same for
all of them: LoRA's $B$ starts at zero, where the adapters are the identity.

```python
{
    0: {"params": ["role:adapter & layers:7"], "max_transition_epoch": 1},
    1: {"params": ["role:adapter & layers:8:"], "max_transition_epoch": 2},
    2: {
        "params": ["role:latent:sae"],
        "promote": {"role:artifact": "latent:sae"},
        "refreeze": ["role:adapter"],
        "lr": 1e-5,
    },
}
```

The staged run keeps checkpoints, and a tuning scheduler that does so restores the best one, by validation loss, at every
stage boundary unless told otherwise. That would give this run alone a choice made on validation data, and can undo a
whole stage, so it is turned off here (`restore_best=False`): each stage goes on from where the last one ended.

After every epoch the test blocks are scored one by one, so two runs are compared block against block.

In [6]:
# @title 3: Run them { display-mode: "form" }
NEXT, LATER = f"role:adapter & layers:{AFTER[0]}", f"role:adapter & layers:{AFTER[1]}:"
SCHEDULES = {
    "next layer": (ADAPTER_LR, {0: {"params": [NEXT]}}),
    "all later layers": (ADAPTER_LR, {0: {"params": [f"role:adapter & layers:{AFTER[0]}:"]}}),
    "the SAE": (SAE_LR, {0: {"params": ["role:latent:sae"]}}),
    "staged": (
        ADAPTER_LR,
        {
            0: {"params": [NEXT], "max_transition_epoch": 1},
            1: {"params": [LATER], "max_transition_epoch": 2},
            2: {
                "params": ["role:latent:sae"],
                "promote": {"role:artifact": "latent:sae"},
                "refreeze": ["role:adapter"],
                "lr": SAE_LR,
            },
        },
    ),
}
GROUPS = {"adapters": ".lora_", "SAE": "latent_models.sae."}


def group_of(name: str) -> str:
    return next((group for group, key in GROUPS.items() if key in name), "base")


@torch.no_grad()
def test_losses(forward, datamodule) -> torch.Tensor:
    """Every test block's loss under ``forward``."""
    losses = []
    for batch in datamodule.test_dataloader():
        ids = batch["input_ids"].to(DEVICE)
        losses.append(block_losses(forward(input_ids=ids).logits, ids))
    return torch.cat(losses).cpu()


def run(name: str, checkpoint_dir: Path | None = None) -> dict:
    lr, schedule = SCHEDULES[name]
    torch.manual_seed(SEED)  # before the build, so every run draws the same LoRA A
    module = (session := build_session(lr, TunedSAEModule if name == "the SAE" else AdaptedModule)).module
    sizes = {n: p.numel() for n, p in module.named_parameters()}
    initial = {n: p.detach().clone() for n, p in module.named_parameters()}
    result = {"epochs": [], "start": None, "base": None}

    class RecordEpochs:
        def on_train_start(self, ctx) -> None:
            module.eval()
            result["base"] = test_losses(TEACHER, module.datamodule)
            result["start"] = test_losses(module, module.datamodule)
            module.train()

        def on_train_epoch_end(self, ctx, metrics) -> None:
            module.eval()
            thawed = [n for n, p in module.named_parameters() if p.requires_grad]
            result["epochs"].append(
                {
                    "stage": fts.tuning_scheduler.curr_depth,
                    "thawed": thawed,
                    "n_params": sum(sizes[n] for n in thawed),
                    "losses": test_losses(module, module.datamodule),
                    "metrics": {k: float(v) for k, v in (metrics or {}).items()},
                }
            )
            module.train()

    kwargs = {}
    if checkpoint_dir is not None:
        kwargs["restore_best"] = False  # each stage goes on from where the last ended, as the one-entry runs do
        kwargs["checkpoint"] = {
            "class_path": "finetuning_scheduler.core.checkpoint.BestCheckpoint",
            # every checkpoint, of the tensors the schedule names
            "init_args": {"dirpath": str(checkpoint_dir), "monitor": "val_loss", "save_top_k": -1, "include": "scheduled"},
        }
    fts = FTSCoreCallback(schedule, tuning_scheduler_kwargs=kwargs)
    SessionRunner(
        run_cfg=SessionRunnerCfg(
            it_session=session,
            max_epochs=EPOCHS,
            limit_train_batches=STEPS_PER_EPOCH,
            limit_val_batches=-1,
            callbacks=[RecordEpochs(), fts],  # the recorder first, so it reads each epoch before a transition
        )
    ).train()
    assert fts.tuning_scheduler.curr_depth == len(schedule) - 1, "every stage must run"
    result["changed"] = sorted(n for n, p in module.named_parameters() if not torch.equal(initial[n], p))
    return result


scratch = tempfile.TemporaryDirectory()  # removed with the kernel
checkpoint_dir = Path(scratch.name)
results = {}
for name in RUNS:
    results[name] = run(name, checkpoint_dir if name == "staged" else None)
    gc.collect()  # one session in memory at a time
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


... (55 outputs left out)


Epoch 1 end: {'train_kl': 0.1235, 'val_loss': 3.5204}



Epoch 2 end: {'train_kl': 0.1124, 'val_loss': 3.5069}


[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule


## 4. How Much of the Gap Each Run Closed

The first table follows each run on the test blocks. A run's standard error is over blocks, of each block's share of the
gap. The second compares runs block against block at the end of the same budget: how much lower one run's loss is than
another's, and on how many blocks.

In [7]:
# @title 4: The gap, run by run { display-mode: "form" }
first = next(iter(results.values()))
l_base, l_sae = first["base"], first["start"]
gap = (l_sae - l_base).mean()
assert not torch.equal(l_sae, l_base), "the SAE is spliced in"
if TEST_BLOCKS >= 256:  # fewer blocks than this measure the gap too noisily to have a sign
    assert gap > 0, "splicing the SAE costs loss"
for name, r in results.items():
    assert torch.equal(r["base"], l_base), name
    assert torch.allclose(r["start"], l_sae, atol=1e-4), f"{name}: every run starts from the same spliced model"


def closed(losses: torch.Tensor) -> tuple[float, float]:
    """The share of the gap closed on the test blocks, and its standard error over blocks."""
    share = (l_sae - losses) / gap
    return share.mean().item(), (share.std() / len(share) ** 0.5).item()


rows = []
for name, r in results.items():
    for i, epoch in enumerate(r["epochs"], 1):
        share, se = closed(epoch["losses"])
        groups = Counter(group_of(n) for n in epoch["thawed"])
        steps = i * STEPS_PER_EPOCH
        rows.append(
            [name, i, epoch["stage"], ", ".join(f"{k} ({v})" for k, v in groups.items()), f"{epoch['n_params'] / 1e6:.2f} M",
             f"{steps:,}", f"{steps * BATCH_SIZE * CONTEXT / 1e6:.2f} M", f"{epoch['losses'].mean():.4f}", f"{share:.1%} ± {se:.1%}"]
        )
display_notebook_table(
    rows,
    ["Run", "Epoch", "Stage", "Thawed (tensors)", "Parameters", "Backward passes", "Tokens", "Test loss", "Gap closed"],
    title="Test loss and gap closed",
    caption=f"L_BASE {l_base.mean():.4f}, L_SAE {l_sae.mean():.4f}, gap {gap:.4f} nats, over {len(l_base)} test blocks.",
)

final = {name: r["epochs"][-1]["losses"] for name, r in results.items()}
pairs = [("all later layers", "next layer"), ("staged", "next layer"), ("staged", "all later layers"),
         ("the SAE", "all later layers")]
rows = []
for a, b in pairs:
    if a in final and b in final:
        diff = final[b] - final[a]  # positive: a's loss is lower
        rows.append([f"{a} vs {b}", f"{diff.mean() / gap:+.1%} ± {diff.std() / len(diff) ** 0.5 / gap:.1%}",
                     f"{(diff > 0).float().mean():.0%}"])
display_notebook_table(
    rows,
    ["Comparison", "Gap closed, difference", "Blocks on which the first is lower"],
    title="Paired by test block, after the whole budget",
)
if {"next layer", "all later layers"} <= final.keys() and TEST_BLOCKS >= 256:
    one, every = closed(final["next layer"])[0], closed(final["all later layers"])[0]
    print(f"one layer reaches {one / every:.0%} of what every later layer does (the paper: about 88%)")

Run,Epoch,Stage,Thawed (tensors),Parameters,Backward passes,Tokens,Test loss,Gap closed
next layer,1,0,adapters (8),0.79 M,400,0.82 M,3.8737,44.5% ± 2.4%
next layer,2,0,adapters (8),0.79 M,800,1.64 M,3.8616,51.6% ± 2.7%
next layer,3,0,adapters (8),0.79 M,"1,200",2.46 M,3.8604,52.3% ± 2.7%
all later layers,1,0,adapters (40),3.93 M,400,0.82 M,3.8390,64.9% ± 3.2%
all later layers,2,0,adapters (40),3.93 M,800,1.64 M,3.8255,72.9% ± 3.5%
all later layers,3,0,adapters (40),3.93 M,"1,200",2.46 M,3.8220,75.0% ± 3.5%
the SAE,1,0,SAE (4),50.37 M,400,0.82 M,3.8825,39.3% ± 2.1%
the SAE,2,0,SAE (4),50.37 M,800,1.64 M,3.8741,44.3% ± 2.4%
the SAE,3,0,SAE (4),50.37 M,"1,200",2.46 M,3.8663,48.9% ± 2.5%
staged,1,0,adapters (8),0.79 M,400,0.82 M,3.8737,44.5% ± 2.4%


Comparison,"Gap closed, difference",Blocks on which the first is lower
all later layers vs next layer,+22.7% ± 1.0%,90%
staged vs next layer,+4.6% ± 0.5%,67%
staged vs all later layers,-18.1% ± 1.0%,16%
the SAE vs all later layers,-26.1% ± 1.5%,23%


one layer reaches 70% of what every later layer does (the paper: about 88%)


## 5. What Each Stage Tuned, and What Its Checkpoint Holds

The schedules named no parameter, so what they thawed is read from the run. The staged run also kept the checkpoint of
every epoch (`BestCheckpoint` with `save_top_k=-1`), of the scheduled tensors alone. The SAE is in none of them until its
promotion, since a published dictionary is reloaded from its source, and in the one saved after it, whose values are the
run's own.

In [8]:
# @title 5: The guarantees, as observed { display-mode: "form" }
for name, r in results.items():
    tuned = {n for epoch in r["epochs"] for n in epoch["thawed"]}
    differ = sorted(set(r["changed"]) ^ tuned)
    assert not differ, f"{name}: what changed is what a stage thawed, not so for {len(differ)} tensors: {differ[:6]}"
    assert "base" not in {group_of(n) for n in tuned}, f"{name}: the original model's weights are never tuned"
if "staged" in results:
    groups = [{group_of(n) for n in epoch["thawed"]} for epoch in results["staged"]["epochs"]]
    assert groups == [{"adapters"}, {"adapters"}, {"SAE"}], groups
    first_layer = {n for n in results["staged"]["epochs"][0]["thawed"]}
    assert first_layer and all(f".h.{AFTER[0]}." in n for n in first_layer), "stage 0 adapts the next block alone"
    saved = []
    for path in sorted(checkpoint_dir.glob("*.pt"), key=lambda p: p.stat().st_mtime):
        payload = load_payload(path)
        held = {group_of(k) for k in payload["model"]}
        saved.append([path.name, *("yes" if g in held else "no" for g in GROUPS)])
    display_notebook_table(
        saved,
        ["Checkpoint", *GROUPS],
        title="What each of the staged run's checkpoints holds, in the order it wrote them",
    )
    holds_sae = [row[-1] for row in saved]
    assert holds_sae == ["no", "no", "yes"], f"the SAE is checkpointed once promoted: {saved}"

Checkpoint,adapters,SAE
best-1-3b01bffa.pt,yes,no
best-2-614048ab.pt,yes,no
best-3-2966545d.pt,yes,yes
